# autoresearch-inference: Experiment Analysis

Visualize the progression of inference optimization experiments.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load results
df = pd.read_csv('results.tsv', sep='\t')
print(f'Total experiments: {len(df)}')
print(f'Kept: {len(df[df.status == "keep"])}')
print(f'Discarded: {len(df[df.status == "discard"])}')
print(f'Crashed: {len(df[df.status == "crash"])}')
df.head(20)

In [ ]:
# Plot tok/s progression (all experiments)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'keep': '#2ecc71', 'discard': '#e74c3c', 'crash': '#95a5a6'}
c = df['status'].map(colors)

axes[0].scatter(range(len(df)), df['tok_s'], c=c, s=40, alpha=0.7)
axes[0].set_xlabel('Experiment #')
axes[0].set_ylabel('tok/s')
axes[0].set_title('All Experiments: tok/s')
axes[0].grid(True, alpha=0.3)

# Plot only kept experiments (monotonically increasing)
kept = df[df.status == 'keep'].reset_index(drop=True)
axes[1].plot(range(len(kept)), kept['tok_s'], 'o-', color='#2ecc71', markersize=6)
axes[1].set_xlabel('Kept Experiment #')
axes[1].set_ylabel('tok/s')
axes[1].set_title('Kept Experiments: tok/s Progression')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('progress.png', dpi=150)
plt.show()

if len(kept) > 1:
    baseline = kept.iloc[0]['tok_s']
    best = kept.iloc[-1]['tok_s']
    print(f'\nBaseline: {baseline:.2f} tok/s')
    print(f'Best:     {best:.2f} tok/s')
    print(f'Speedup:  {best/baseline:.2f}x')

In [ ]:
# VRAM usage over experiments
fig, ax = plt.subplots(figsize=(10, 4))
valid = df[df.peak_vram_gb > 0]
c = valid['status'].map(colors)
ax.scatter(range(len(valid)), valid['peak_vram_gb'], c=c, s=40, alpha=0.7)
ax.axhline(y=31, color='red', linestyle='--', alpha=0.5, label='VRAM limit (31 GB)')
ax.set_xlabel('Experiment #')
ax.set_ylabel('Peak VRAM (GB)')
ax.set_title('Peak VRAM Usage')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Summary table of kept experiments
if len(kept) > 0:
    print('Kept experiments (improvements):')
    print('=' * 80)
    for i, row in kept.iterrows():
        print(f"  {row['commit']}  {row['tok_s']:7.2f} tok/s  "
              f"{row['peak_vram_gb']:5.1f} GB  {row['description']}")